# CNN 128 — Redes Neurais (TrashNet)

Experimentos `cnn_128_sem` e `cnn_128_com` (integrante 3) e comparação com o MLP **na validação**.

Antes: **Ambiente de execução → Alterar tipo → GPU (T4)**.

Tudo fica salvo no seu Google Drive. Se o Colab cair, rode as células de novo: o treino continua da última época.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive
!test -d neural-network-project || git clone https://github.com/matheusvsmp/neural-network-project.git
%cd neural-network-project
!git pull
!ls

Coloque `common.py` e `03_cnn.py` na pasta `neural-network-project` do seu Drive (se ainda não estiverem no repositório).

In [ ]:
# Baixa o TrashNet (só na primeira vez)
!test -d dataset-resized || (git clone --depth 1 https://github.com/garythung/trashnet.git /content/trashnet && unzip -q /content/trashnet/data/dataset-resized.zip -d . && rm -rf __MACOSX)
!ls dataset-resized

In [ ]:
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))

## Treino

Integrante 4 (resolução): `!python 03_cnn.py --exp cnn_256_sem cnn_512_sem` (se faltar memória, acrescente `--batch_size 16` e justifique no artigo).

In [ ]:
!python 03_cnn.py

In [ ]:
import json
for nome in ['cnn_128_sem', 'cnn_128_com']:
    m = json.load(open(f'historicos/{nome}.json'))['meta']
    print(nome, '| val F1 macro', round(m['val_f1_macro'], 3), '| val acc', round(m['val_accuracy'], 3),
          '| época salva', m['epoca_do_modelo_salvo'], 'de', m['epocas_rodadas'],
          '| min', round(m['tempo_total_treino_s']/60, 1), '| s/época', m['tempo_medio_por_epoca_s'],
          '| MB', m['tamanho_modelo_salvo_mb'], '| GPU', m['hardware']['gpu'])

In [ ]:
from IPython.display import Image, display
for nome in ['cnn_128_sem', 'cnn_128_com']:
    display(Image(f'figuras/curvas/{nome}.png'))

## Comparação MLP × CNN (só VALIDAÇÃO)

O conjunto de teste **não** é usado aqui — ele fica para o `05_resultados.py`, no final. Estas tabelas servem para o texto sobre CNN e pesos de classe.

In [ ]:
import pandas as pd
from pathlib import Path

NOMES = ['mlp_128_sem', 'mlp_128_com', 'cnn_128_sem', 'cnn_128_com']
linhas = []
for nome in NOMES:
    p = Path(f'historicos/{nome}.json')
    if not p.exists():
        print('ainda não treinado:', nome)
        continue
    m = json.loads(p.read_text(encoding='utf-8'))['meta']
    linhas.append({'experimento': nome, 'val_f1_macro': m['val_f1_macro'], 'val_acc': m['val_accuracy'],
                   'época salva': m['epoca_do_modelo_salvo'], 'épocas': m['epocas_rodadas'],
                   'parâmetros': m['parametros_treinaveis'], 'MB': m['tamanho_modelo_salvo_mb'],
                   'tempo total (min)': round(m['tempo_total_treino_s'] / 60, 1),
                   's/época': m['tempo_medio_por_epoca_s']})
tabela = pd.DataFrame(linhas).set_index('experimento')
tabela.round(3)

### Efeito dos pesos de classe por classe (validação)

Precisão, recall e F1 por classe de cada modelo na validação, e a diferença **com − sem pesos** em cada arquitetura (atenção à classe **trash**).

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import precision_recall_fscore_support
from common import CLASSES, carregar_split, caminhos_e_rotulos, criar_dataset

split = carregar_split()
_, y_val = caminhos_e_rotulos(split, 'val')
ds_val = criar_dataset(split, 'val', 128)

por_classe = {}
for nome in NOMES:
    p = Path(f'modelos/{nome}.keras')
    if not p.exists():
        continue
    modelo = tf.keras.models.load_model(p, compile=False)
    y_pred = np.argmax(modelo.predict(ds_val, verbose=0), axis=1)
    pr, rc, f1, sup = precision_recall_fscore_support(y_val, y_pred, labels=range(len(CLASSES)), zero_division=0)
    por_classe[nome] = pd.DataFrame({'precisão': pr, 'recall': rc, 'F1': f1, 'suporte': sup}, index=CLASSES)
    print(f'\n{nome}')
    display(por_classe[nome].round(3))

In [ ]:
for arq in ['mlp', 'cnn']:
    sem, com = f'{arq}_128_sem', f'{arq}_128_com'
    if sem in por_classe and com in por_classe:
        dif = (por_classe[com][['precisão', 'recall', 'F1']] - por_classe[sem][['precisão', 'recall', 'F1']]).round(3)
        print(f'\n{arq.upper()}: com pesos − sem pesos (validação)')
        display(dif.style.background_gradient(cmap='RdYlGn', vmin=-0.3, vmax=0.3))

### Curvas de aprendizagem lado a lado

In [ ]:
for nome in NOMES:
    p = Path(f'figuras/curvas/{nome}.png')
    if p.exists():
        display(Image(str(p)))